In [ ]:
# %pip install -q transformers accelerate sentencepiece scikit-learn scipy pandas numpy tqdm


In [1]:
import os, gc, random, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import cohen_kappa_score
from scipy.optimize import minimize
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_cosine_schedule_with_warmup

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [2]:
class CFG:
    seed = 42
    model_name = "microsoft/deberta-v3-small"
    max_length = 768
    epochs = 2
    n_folds = 5
    train_batch_size = 8
    valid_batch_size = 16
    grad_accum = 2
    lr = 2e-5
    weight_decay = 0.01
    warmup_ratio = 0.06
    max_grad_norm = 1.0
    num_workers = 0
    amp = True
    output_dir = Path("./aes2_outputs")

CFG.output_dir.mkdir(parents=True, exist_ok=True)

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True

seed_everything(CFG.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))


device: cuda
NVIDIA GeForce RTX 4070


In [3]:
def find_competition_file(filename):
    roots = [
        Path("/kaggle/input/learning-agency-lab-automated-essay-scoring-2"),
        Path("./learning-agency-lab-automated-essay-scoring-2/data/learning-agency-lab-automated-essay-scoring-2"),
        Path("./input/learning-agency-lab-automated-essay-scoring-2"),
        Path("./input"),
        Path("."),
    ]
    for root in roots:
        p = root / filename
        if p.exists():
            return p
    matches = list(Path(".").rglob(filename))
    if matches:
        return matches[0]
    raise FileNotFoundError(f"Could not find {filename}. Put train.csv/test.csv in the notebook folder or update the paths.")

train_path = find_competition_file("train.csv")
test_path = find_competition_file("test.csv")

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
train["full_text"] = train["full_text"].fillna("")
test["full_text"] = test["full_text"].fillna("")
train["label"] = train["score"].astype(float)

print(train_path, train.shape)
print(test_path, test.shape)
print(train["score"].value_counts().sort_index())


learning-agency-lab-automated-essay-scoring-2\data\learning-agency-lab-automated-essay-scoring-2\train.csv (15576, 4)
learning-agency-lab-automated-essay-scoring-2\data\learning-agency-lab-automated-essay-scoring-2\test.csv (1731, 2)
score
1    1124
2    4249
3    5629
4    3563
5     876
6     135
Name: count, dtype: int64


In [4]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

class EssayDataset(Dataset):
    def __init__(self, texts, labels=None):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=CFG.max_length,
            padding=False,
            return_tensors=None,
        )
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in enc.items()}
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item

class Collator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer

    def __call__(self, batch):
        labels = None
        if "labels" in batch[0]:
            labels = torch.stack([x.pop("labels") for x in batch])
        batch = self.tokenizer.pad(batch, padding=True, return_tensors="pt")
        if labels is not None:
            batch["labels"] = labels
        return batch

collate_fn = Collator(tokenizer)


config.json:   0%|          | 0.00/578 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

In [5]:
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true.astype(int), y_pred.astype(int), weights="quadratic")

def apply_thresholds(preds, thresholds):
    thresholds = np.sort(np.array(thresholds))
    return np.clip(np.digitize(preds, thresholds) + 1, 1, 6).astype(int)

def optimize_thresholds(y_true, preds):
    def loss(thresholds):
        return -qwk(y_true, apply_thresholds(preds, thresholds))
    result = minimize(loss, x0=[1.5, 2.5, 3.5, 4.5, 5.5], method="Nelder-Mead")
    return np.sort(result.x)


In [6]:
def build_model():
    model = AutoModelForSequenceClassification.from_pretrained(
        CFG.model_name,
        num_labels=1,
        problem_type="regression",
    )
    model.config.hidden_dropout_prob = 0.1
    model.config.attention_probs_dropout_prob = 0.1
    return model

loss_fn = nn.SmoothL1Loss()

def train_one_epoch(model, loader, optimizer, scheduler, scaler):
    model.train()
    losses = []
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, leave=False)
    for step, batch in enumerate(pbar):
        labels = batch.pop("labels").to(device)
        batch = {k: v.to(device) for k, v in batch.items()}

        use_amp = CFG.amp and device.type == "cuda"
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
            preds = model(**batch).logits.squeeze(-1)
            loss = loss_fn(preds, labels) / CFG.grad_accum

        if scaler is not None:
            scaler.scale(loss).backward()
        else:
            loss.backward()

        if (step + 1) % CFG.grad_accum == 0 or (step + 1) == len(loader):
            if scaler is not None:
                scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
            if scaler is not None:
                scaler.step(optimizer)
                scaler.update()
            else:
                optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item() * CFG.grad_accum)
        pbar.set_postfix(loss=np.mean(losses[-50:]))
    return float(np.mean(losses))

@torch.no_grad()
def predict(model, loader):
    model.eval()
    preds = []
    for batch in tqdm(loader, leave=False):
        batch.pop("labels", None)
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch).logits.squeeze(-1).detach().cpu().numpy()
        preds.append(out)
    return np.concatenate(preds)


In [8]:
skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
train["fold"] = -1
for fold, (_, valid_idx) in enumerate(skf.split(train, train["score"])):
    train.loc[valid_idx, "fold"] = fold

oof = np.zeros(len(train), dtype=np.float32)
test_preds = np.zeros(len(test), dtype=np.float32)
fold_scores = []

test_ds = EssayDataset(test["full_text"].tolist())
test_loader = DataLoader(
    test_ds,
    batch_size=CFG.valid_batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=CFG.num_workers,
    pin_memory=(device.type == "cuda"),
)

for fold in range(CFG.n_folds):
    print(f"\nfold {fold}")
    trn_idx = train.index[train["fold"] != fold].to_numpy()
    val_idx = train.index[train["fold"] == fold].to_numpy()

    train_ds = EssayDataset(train.loc[trn_idx, "full_text"].tolist(), train.loc[trn_idx, "label"].values)
    valid_ds = EssayDataset(train.loc[val_idx, "full_text"].tolist(), train.loc[val_idx, "label"].values)

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG.train_batch_size,
        shuffle=True,
        collate_fn=collate_fn,
        num_workers=CFG.num_workers,
        pin_memory=(device.type == "cuda"),
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=CFG.valid_batch_size,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=CFG.num_workers,
        pin_memory=(device.type == "cuda"),
    )

    model = build_model().to(device)

    # Keep weights in FP32. AMP handles FP16 inside autocast.
    model.float()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.lr,
        weight_decay=CFG.weight_decay
    )

    total_steps = (len(train_loader) // CFG.grad_accum + 1) * CFG.epochs
    warmup_steps = int(total_steps * CFG.warmup_ratio)

    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        warmup_steps,
        total_steps
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(CFG.amp and device.type == "cuda")
    )

    best_score = -1
    best_preds = None
    best_path = CFG.output_dir / f"fold{fold}.pt"

    for epoch in range(CFG.epochs):
        loss = train_one_epoch(model, train_loader, optimizer, scheduler, scaler)
        val_preds = predict(model, valid_loader)
        val_round = np.clip(np.rint(val_preds), 1, 6).astype(int)
        score = qwk(train.loc[val_idx, "score"].values, val_round)
        print(f"epoch {epoch+1}: loss={loss:.5f} qwk={score:.5f}")
        if score > best_score:
            best_score = score
            best_preds = val_preds.copy()
            torch.save(model.state_dict(), best_path)

    model.load_state_dict(torch.load(best_path, map_location=device))
    oof[val_idx] = best_preds
    test_preds += predict(model, test_loader) / CFG.n_folds
    fold_scores.append(best_score)

    del model, optimizer, scheduler, scaler, train_loader, valid_loader, train_ds, valid_ds
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

print("fold qwk:", fold_scores)
print("mean qwk:", np.mean(fold_scores))



fold 0


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifier.bias         

  0%|          | 0/1558 [00:00<?, ?it/s]

  0%|          | 0/195 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
y_true = train["score"].values.astype(int)
raw_oof = np.clip(np.rint(oof), 1, 6).astype(int)
thresholds = optimize_thresholds(y_true, oof)
opt_oof = apply_thresholds(oof, thresholds)

print("raw oof qwk:", qwk(y_true, raw_oof))
print("opt oof qwk:", qwk(y_true, opt_oof))
print("thresholds:", thresholds)


In [ ]:
submission = pd.DataFrame({
    "essay_id": test["essay_id"],
    "score": apply_thresholds(test_preds, thresholds),
})

out_path = Path("/kaggle/working/submission.csv") if Path("/kaggle/working").exists() else Path("submission.csv")
submission.to_csv(out_path, index=False)
print(out_path.resolve())
display(submission.head())
submission["score"].value_counts().sort_index()
